In [0]:
%sql

USE CATALOG workspace;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS project_20a_healthcare;

In [0]:
%sql

USE SCHEMA project_20a_healthcare;

In [0]:
%sql

SELECT CURRENT_CATALOG(), CURRENT_SCHEMA();

In [0]:
%sql

CREATE OR REPLACE TABLE FACT_PATIENTS_ENCOUNTER
(
    ENCOUNTER_ID STRING,
    PATIENT_ID STRING,
    ADMIT_DATE_KEY BIGINT,
    DISCHARGE_DATE_KEY BIGINT,
    BILLING_DATE_KEY BIGINT,
    PRIMARY_DIAG_ID STRING,
    TOTAL_COST DECIMAL(10,2)
)
USING DELTA 
PARTITIONED BY (ADMIT_DATE_KEY);

In [0]:
%sql

DESCRIBE DETAIL FACT_PATIENTS_ENCOUNTER;

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS project_20a_healthcare.project_20a_data;

In [0]:
%sql

SHOW VOLUMES IN project_20a_healthcare;    

In [0]:
%sql

SELECT *
FROM read_files(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/raw_encounters.json',
    FORMAT => 'json'
);

In [0]:
%sql

INSERT INTO FACT_PATIENTS_ENCOUNTER(
    ENCOUNTER_ID,
    PATIENT_ID,
    ADMIT_DATE_KEY,
    DISCHARGE_DATE_KEY,
    BILLING_DATE_KEY,
    PRIMARY_DIAG_ID,
    TOTAL_COST
)
SELECT  encounter_id,
        patient_id,
        CAST(admit_date_key AS BIGINT),
        CAST(discharge_date_key AS BIGINT),
        CAST(billing_date_key AS BIGINT),
        primary_diag_id,
        CAST(total_cost AS DECIMAL(10,2))
FROM READ_FILES(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/raw_encounters.json',
    FORMAT => 'json'
);

In [0]:
%sql

SELECT *
FROM FACT_PATIENTS_ENCOUNTER
ORDER BY ENCOUNTER_ID;

In [0]:
%sql

SELECT *
FROM read_files(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_date.json',
    FORMAT => 'json'
);

In [0]:
%sql

CREATE OR REPLACE VIEW VW_ADMIT_DATE AS
SELECT  DISTINCT FPE.ADMIT_DATE_KEY,
        DD.full_date AS ADMIT_DATE
FROM FACT_PATIENTS_ENCOUNTER FPE
JOIN READ_FILES(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_date.json',
    FORMAT => 'json'
) DD
ON FPE.ADMIT_DATE_KEY=DD.date_key;

In [0]:
%sql

SELECT *
FROM VW_ADMIT_DATE
ORDER BY ADMIT_DATE_KEY;

In [0]:
%sql

CREATE OR REPLACE VIEW VW_DISCHARGE_DATE AS
SELECT  DISTINCT FPE.DISCHARGE_DATE_KEY,
        DD.full_date AS DISCHARGE_DATE
FROM FACT_PATIENTS_ENCOUNTER FPE
JOIN READ_FILES(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_date.json',
    FORMAT => 'json'
) DD
ON FPE.DISCHARGE_DATE_KEY=DD.date_key;

In [0]:
%sql

SELECT *
FROM VW_DISCHARGE_DATE
ORDER BY DISCHARGE_DATE_KEY;

In [0]:
%sql

CREATE OR REPLACE VIEW VW_BILLING_DATE AS
SELECT  DISTINCT FPE.BILLING_DATE_KEY,
        DD.full_date AS BILLING_DATE
FROM FACT_PATIENTS_ENCOUNTER FPE
JOIN READ_FILES(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_date.json',
    FORMAT => 'json'
) DD
ON FPE.BILLING_DATE_KEY=DD.date_key;

In [0]:
%sql

SELECT *
FROM VW_BILLING_DATE
ORDER BY BILLING_DATE_KEY;

In [0]:
%sql

SELECT  FPE.ENCOUNTER_ID,
        VAD.ADMIT_DATE,
        VDD.DISCHARGE_DATE,
        VBD.BILLING_DATE
FROM FACT_PATIENTS_ENCOUNTER FPE
INNER JOIN VW_ADMIT_DATE VAD
ON FPE.ADMIT_DATE_KEY=VAD.ADMIT_DATE_KEY
INNER JOIN VW_DISCHARGE_DATE VDD
ON FPE.DISCHARGE_DATE_KEY=VDD.DISCHARGE_DATE_KEY
INNER JOIN VW_BILLING_DATE VBD
ON FPE.BILLING_DATE_KEY=VBD.BILLING_DATE_KEY
ORDER BY FPE.ENCOUNTER_ID
LIMIT 3;


In [0]:
%sql

SELECT *
FROM READ_FILES(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/bridge_doctors.json',
    FORMAT => 'json'
);

In [0]:
%sql

SELECT  FPE.ENCOUNTER_ID,
        BD.doctor_id AS DOCTOR_ID,
        BD.weight_factor AS WEIGHT_FACTOR,
        FPE.TOTAL_COST*BD.weight_factor AS ATTRIBUTED_COST
FROM FACT_PATIENTS_ENCOUNTER FPE
INNER JOIN READ_FILES(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/bridge_doctors.json',
    FORMAT => 'json'
) BD
ON FPE.ENCOUNTER_ID=BD.encounter_id
WHERE FPE.ENCOUNTER_ID IN ('ENC-801','ENC-802')
ORDER BY FPE.ENCOUNTER_ID,DOCTOR_ID;

In [0]:
%sql

SELECT  *
FROM read_files(
    '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_diagnosis.json',
    FORMAT => 'json'
)
ORDER BY diagnosis_id;

In [0]:
%sql

WITH RECURSIVE diagnosis_tree AS
(
    SELECT
        diagnosis_id,
        diagnosis_name,
        parent_diagnosis_id,
        category_level,
        diagnosis_id AS root_diagnosis_id,
        diagnosis_name AS root_category_name
    FROM read_files(
        '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_diagnosis.json',
        format => 'json'
    )
    WHERE category_level = 1

    UNION ALL

    -- Recursive step: find children of the current node
    SELECT
        D.diagnosis_id,
        D.diagnosis_name,
        D.parent_diagnosis_id,
        D.category_level,
        DT.root_diagnosis_id,
        DT.root_category_name
    FROM diagnosis_tree DT
    INNER JOIN read_files(
        '/Volumes/workspace/project_20a_healthcare/project_20a_data/dim_diagnosis.json',
        format => 'json'
    ) D
        ON D.parent_diagnosis_id = DT.diagnosis_id
)
SELECT
    diagnosis_id,
    diagnosis_name,
    root_category_name
FROM diagnosis_tree
ORDER BY diagnosis_id;

In [0]:
%sql

CREATE VIEW INCOMING_DATA AS
SELECT  'ENC-801' AS ENCOUNTER_ID,   
        'PAT-10' AS PATIENT_ID,
        20260801 AS ADMIT_DATE_KEY,
        20260804 AS DISCHARGE_DATE_KEY,
        20260806 AS BILLING_DATE_KEY,
        'ICD-1110' AS PRIMARY_DIAG_ID,
        CAST(5000.00 AS DECIMAL(10,2)) AS TOTAL_COST
UNION ALL
SELECT
    'ENC-821',
    'PAT-30',
    20260816,
    20260820,
    20260821,
    'ICD-1120',
    CAST(7000.00 AS DECIMAL(10,2));

In [0]:
%sql

MERGE INTO FACT_PATIENTS_ENCOUNTER T
USING INCOMING_DATA S
ON T.ENCOUNTER_ID = S.ENCOUNTER_ID
WHEN MATCHED THEN
UPDATE SET 
         T.PATIENT_ID = S.PATIENT_ID,
         T.ADMIT_DATE_KEY = S.ADMIT_DATE_KEY,
         T.DISCHARGE_DATE_KEY = S.DISCHARGE_DATE_KEY,
         T.BILLING_DATE_KEY = S.BILLING_DATE_KEY,
         T.PRIMARY_DIAG_ID = S.PRIMARY_DIAG_ID,
         T.TOTAL_COST = S.TOTAL_COST
WHEN NOT MATCHED THEN
INSERT (ENCOUNTER_ID,
         PATIENT_ID,
         ADMIT_DATE_KEY,
         DISCHARGE_DATE_KEY,
         BILLING_DATE_KEY,
         PRIMARY_DIAG_ID,
         TOTAL_COST)
VALUES (S.ENCOUNTER_ID,
         S.PATIENT_ID,
         S.ADMIT_DATE_KEY,
         S.DISCHARGE_DATE_KEY,
         S.BILLING_DATE_KEY,
         S.PRIMARY_DIAG_ID,
         S.TOTAL_COST);

In [0]:
%sql
SELECT * FROM FACT_PATIENTS_ENCOUNTER
ORDER BY ENCOUNTER_ID;

In [0]:
%sql

DESCRIBE HISTORY FACT_PATIENTS_ENCOUNTER;

In [0]:
%sql

SELECT ENCOUNTER_ID,
       TOTAL_COST

FROM FACT_PATIENTS_ENCOUNTER VERSION AS OF 1
ORDER BY ENCOUNTER_ID;

In [0]:
%sql

ALTER TABLE FACT_PATIENTS_ENCOUNTER 
ADD COLUMN PATIENT_FEEDBACK_SCORE DOUBLE;

In [0]:
%sql

DESCRIBE FACT_PATIENTS_ENCOUNTER;

In [0]:
%sql

OPTIMIZE FACT_PATIENTS_ENCOUNTER
ZORDER BY (PATIENT_ID,PRIMARY_DIAG_ID);

In [0]:
%sql
SELECT COUNT(*) FROM FACT_PATIENTS_ENCOUNTER;

In [0]:
%sql

DESCRIBE HISTORY FACT_PATIENTS_ENCOUNTER;